<div style="background:linear-gradient(120deg,#0b1f3a 0%,#1d4e89 45%,#0f9d9a 100%); padding:30px 34px; border-radius:14px; color:#ffffff;">
<h1 style="color:#ffffff; margin:0 0 10px 0; font-weight:800;">WEAR@HASCA | Timelines, Graphs and Learned Counts | LB 0.92</h1>
<p style="color:#e8f1f8; font-size:1.05em; margin:0; line-height:1.55;">
Ranked continuation links decoded into eight timeline hypotheses, graph propagation over whitened video embeddings with
learned per-participant activity counts, and a boundary refiner that decides where the timelines' label changes belong.
</p>
</div>

## The competition in brief

| Item | Detail |
|---|---|
| Task | one activity label per one-second window |
| Classes | 19 (18 exercises + null) |
| Development data | 69,326 windows from 22 participants, grouped into five participant folds |
| Test | 12,234 windows |
| Metric | macro-F1 |

The windows are **shuffled**: row order says nothing about time. Each participant performs every exercise for roughly
the same duration, so the order of windows, and how many belong to each exercise, can be recovered from the data.

## Pipeline

```
window model + S3 sensor head + teacher ──► per-window evidence (19 log-probabilities)
                         │
ranked continuation links ──► 8 timeline hypotheses (which window comes next)
                         │
whitened video kNN graph + timeline links ──► propagation ──► learned counts ──► propagation ──► learned counts
                         │
label changes along each timeline ──► 89 boundary features ──► 4-seed refiner ──► labels
```

The first two stages are loaded from saved outputs; everything from the graph onwards runs in this notebook.

## Setup

In [ ]:
import os, sys, glob, time, csv, hashlib, warnings
for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"): os.environ[k] = "4"
warnings.filterwarnings("ignore")
import numpy as np, matplotlib.pyplot as plt, lightgbm as lgb
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11})

DATA = os.path.dirname(glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)[0])
EVID = os.path.dirname(glob.glob("/kaggle/input/**/hasca_pipeline.py", recursive=True)[0])
sys.path.insert(0, EVID)
import hasca_pipeline as hp
T0 = time.time(); elapsed = lambda: f"{time.time() - T0:.0f}s"
L = lambda name: np.load(f"{EVID}/{name}.npz", allow_pickle=False)
pre, teacher, links, test_links = L("strict_oof_pregraph"), L("strong_teacher_t"), L("strict_rank_margin_timeline_links"), L("rank_margin_test_links")
graph, geometry = L("whitened_graph"), L("whitening_geometry")
subjects, folds, y = pre["subject"], pre["fold"], pre["y"].astype(np.int64)
test_subject, ids = test_links["subject"], pre["test_id"]
print(f"development windows {len(y):,} | participants {len(np.unique(subjects))} | test windows {len(ids):,}")

## Window evidence

Three saved heads score every window: a window model on the raw signals, an S3 sensor head, and a teacher bagged from
public models. They are combined as a log-linear blend (0.2 / 0.5 / 0.3).

In [ ]:
train_logp = hp.blend(pre["oof_window_logp"], pre["oof_s3"], teacher["public_bagged_oof"])
test_logp = hp.blend(pre["test_window_logp"], pre["test_s3"], teacher["public_bagged_test"])
for name, lp in (("window model", pre["oof_window_logp"]), ("S3 sensor head", pre["oof_s3"]), ("teacher", teacher["public_bagged_oof"]), ("blend", train_logp)):
    print(f"{name:15s} out-of-fold macro-F1 {hp.macro_f1(y, np.asarray(lp).argmax(1)):.4f}")

## Timelines: which window comes next

Because the windows are shuffled, the strongest signal is not in any single window but in its neighbours in time.
For each window a LambdaRank model scores the candidate successors **within that window's candidate set**, and a
one-to-one assignment turns those scores into chains. One assignment uses the clean scores and seven use fixed
perturbations, giving eight timeline hypotheses. Each link keeps a confidence: the chosen candidate's score minus the
best alternative.

> **A link score needs a competitor.** Asking "which candidate wins for this window" ranks continuations better than
> scoring every pair independently, and the margin over the runner-up is a more honest confidence than the raw score.

The development links can be compared with the true successor:

In [ ]:
succ, true_succ, conf = links["succ"], links["true_succ"], links["score_qn"]
acc = [(succ[k] == true_succ)[true_succ >= 0].mean() for k in range(8)]
agree = (succ == succ[0]).mean(0)
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
ax[0].bar(range(8), acc, color=["#1d4e89"] + ["#0f9d9a"] * 7); ax[0].set_ylim(min(acc) - 0.01, max(acc) + 0.005)
ax[0].set(title="share of links that point to the true next window", xlabel="timeline hypothesis (0 = clean)", ylabel="accuracy")
correct = succ[0] == true_succ
ax[1].hist([conf[0][correct & (true_succ >= 0)], conf[0][~correct & (true_succ >= 0)]], bins=40, stacked=True, color=["#0f9d9a", "#f2a541"], label=["correct link", "wrong link"])
ax[1].set(title="link confidence, clean hypothesis", xlabel="normalised margin over the best alternative"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"windows whose successor is identical in all eight hypotheses: {(succ == succ[0]).all(0).mean():.1%}")

Below, one participant's first clean timeline: the true labels, the window evidence alone, and the graph output that
uses the timeline.

In [ ]:
s = np.unique(subjects)[3]; nxt = succ[0]
has_pred = np.zeros(len(y), bool); has_pred[nxt[nxt >= 0]] = True
heads = [i for i in np.flatnonzero((subjects == s) & ~has_pred)]
chain, node = [], max(heads, key=lambda h: 0)
lengths = {}
for h in heads:
    n, c = h, []
    while n >= 0 and len(c) < 5000: c.append(n); n = nxt[n]
    lengths[h] = c
chain = max(lengths.values(), key=len)[:600]
rows = np.stack([y[chain], train_logp[chain].argmax(1), graph["probabilities"][chain].argmax(1)])
fig, ax = plt.subplots(figsize=(14, 2.6))
ax.imshow(rows, aspect="auto", cmap="tab20", interpolation="nearest", vmin=0, vmax=19)
ax.set_yticks(range(3)); ax.set_yticklabels(["true label", "window evidence", "after graph"])
ax.set(title=f"participant {s}: {len(chain)} consecutive windows along the clean timeline", xlabel="position along the timeline (seconds)")
plt.tight_layout(); plt.show()

## Graph propagation and learned counts

Each test participant gets a kNN graph over 768-d video embeddings, joined with the timeline links, and the window
evidence is propagated over it (five fixed configurations, averaged). The embeddings are **centred per participant and
whitened** (power 0.5, kNN temperature 0.3, following [goodpjw2008's](https://www.kaggle.com/code/goodpjw2008/wear-hasca-learned-links-counts-lb-0-92942)
whitening option) before the kNN search, which stops a few dominant directions from deciding every neighbourhood.

The graph output then feeds a ridge model that predicts how many windows each exercise occupies for that participant.
Probabilities are rescaled to those counts, propagated again, and counted again: two passes.

In [ ]:
white_test, _ = hp.apply_white(pre["test_emb"], test_subject, geometry["W"])
dd = dict(logp=test_logp, emb=white_test, grp=test_subject, sbj=test_subject, sets={})
count_model_1 = hp.fit_count_model(graph["P0"], train_logp, subjects, y)      # trained on the development graph outputs
count_model_2 = hp.fit_count_model(graph["P1"], train_logp, subjects, y)

P0 = hp.bag_graph(dd, test_links)                                             # pass 1: propagate over all eight timelines
k1, counts_1 = hp.predict_counts(count_model_1, P0, test_logp, test_subject)
P1 = hp.bag_graph(dict(dd, targets=hp.count_targets(k1, counts_1, test_subject)), test_links)   # pass 2: propagate towards learned counts
k2, counts_2 = hp.predict_counts(count_model_2, P1, test_logp, test_subject)
Q_test = hp.calibrated_to_counts(P1, test_subject, hp.count_targets(k2, counts_2, test_subject))
print("graph and counts done", elapsed())

In [ ]:
f1_evidence = hp.macro_f1(y, train_logp.argmax(1)); f1_graph = hp.macro_f1(y, graph["probabilities"].argmax(1))
per_class = lambda pred: [2 * ((pred == c) & (y == c)).sum() / max((pred == c).sum() + (y == c).sum(), 1) for c in range(19)]
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
ax[0].bar(np.arange(19) - 0.2, per_class(train_logp.argmax(1)), 0.4, color="#9bb7c9", label=f"window evidence ({f1_evidence:.4f})")
ax[0].bar(np.arange(19) + 0.2, per_class(graph["probabilities"].argmax(1)), 0.4, color="#1d4e89", label=f"after graph + counts ({f1_graph:.4f})")
ax[0].set(title="per-class F1 on development folds", xlabel="class (0 = null)", ylabel="F1"); ax[0].legend(loc="lower right")
ax[1].scatter(graph["true_counts"], graph["counts_pass2"], s=8, color="#0f9d9a"); lim = [graph["true_counts"].min(), graph["true_counts"].max()]
ax[1].plot(lim, lim, "k--", lw=1); ax[1].set(title="learned counts vs true counts (development, cross-fitted)", xlabel="true windows per exercise", ylabel="predicted")
plt.tight_layout(); plt.show()

## Boundary refiner

Most remaining errors sit at the edges of activities, where a timeline switches label. For every label change along
every hypothesis, the windows within a few steps of the change are proposed for flipping to the neighbouring label.
Each (window, alternative label) proposal gets 89 label-free features, averaged over the hypotheses that propose it:
evidence margins, neighbour agreement, link confidence, and how consistently the eight timelines agree around it.

Four LightGBM models (fixed seeds, 300 rounds) score the proposals; a window flips when the averaged score passes 0.5.

> **Average the timeline uncertainty before learning boundary changes.** Proposals supported by several hypotheses
> were far more reliable than any single timeline's suggestion, so support and agreement enter as features rather
> than as a vote.

In [ ]:
keys, X = hp.boundary_rows(graph["probabilities"], pre["oof_window_logp"], pre["oof_s3"], pre["oof_emb"], links, subjects)
test_keys, X_test = hp.boundary_rows(Q_test, pre["test_window_logp"], pre["test_s3"], pre["test_emb"], test_links, test_subject)
target = (y[keys[:, 0]] == keys[:, 1]).astype(np.int8)
print(f"development proposals {len(keys):,} ({target.mean():.1%} correct flips) | test proposals {len(test_keys):,} | features {X.shape[1]}")

In [ ]:
train_set = lgb.Dataset(X, label=target)
scores = np.mean([lgb.train(dict(hp.REFINER_PARAMS, seed=s), train_set, num_boost_round=300).predict(X_test, num_threads=4) for s in hp.REFINER_SEEDS], 0)
labels = hp.decode(Q_test.argmax(1), test_keys, scores)
print(f"windows changed by the refiner: {(labels != Q_test.argmax(1)).sum()} of {len(labels):,}", elapsed())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.4))
ax.hist(scores, bins=60, color="#1d4e89", log=True); ax.axvline(0.5, ls="--", c="k", lw=1)
ax.set(title="refiner score for each test proposal (log scale); proposals right of the line flip", xlabel="P(flip is correct)")
plt.tight_layout(); plt.show()

## Submission

In [ ]:
sample = list(csv.reader(open(f"{DATA}/sample_submission.csv")))
assert sample[0] == ["id", "target_feature"] and [int(r[0]) for r in sample[1:]] == ids.tolist()
with open("submission.csv", "w", newline="") as h:
    w = csv.writer(h); w.writerow(sample[0]); w.writerows(zip(ids.tolist(), labels.tolist()))
reference = np.array([int(r[1]) for r in list(csv.reader(open(f"{EVID}/reference_submission.csv")))[1:]])
print(f"rows {len(labels):,} | classes {labels.min()}-{labels.max()} | labels differing from the reference submission: {(labels != reference).sum()} | runtime {elapsed()}")

## Validation: cross-fitting has to be traced through every model

The refiner never sees its held-out participants, but that alone does not make the held-out score honest. A count
model that built *another* fold's training features may itself have been trained on the held-out fold, and its graph
then reaches the refiner through the second count pass. Changing only the held-out fold's count targets moved
**32 held-out predictions** in the original recipe. Excluding both the outer fold and each profile's own fold in both
count passes made the result exactly invariant to that change.

Under that repaired protocol, whitening with the kNN temperature raised development macro-F1 from **0.92899 to
0.93203** (+0.0030, all five participant folds positive), while it changed 88 test labels and moved the public score by
only +0.0001. These figures remain conditional on upstream choices made with the same folds, so they are not a fully
nested estimate.

## Experiment log

Each change was evaluated in isolation against the preceding configuration; changes made after the whitened geometry
were evaluated against that configuration. Rows in grey did not improve the score.

| Experiment | Public LB | Effect | Derived conclusion |
|---|---:|---:|---|
| Reproduction of the public baseline | 0.6646 | | Starting point |
| Sensor specialists with a per-participant activity minimum | 0.6894 | +0.025 | Each exercise has a minimum duration |
| VideoMAE similarity graph with a 90 s activity quota | 0.7775 | +0.088 | Similar windows share labels within a participant |
| Supervised video head, fixed global decoder | 0.8092 | +0.032 | Video embeddings carry class information directly |
| CatBoost–LightGBM blend with video prototypes | 0.8280 | +0.019 | Model diversity helps the window evidence |
| Cross-sensor VideoMAE graph with model affinity | 0.8365 | +0.009 | Graph edges should agree with the evidence |
| Timeline reconstruction on the public stack (Jiwei Liu, Briano) | 0.9077 | +0.071 | Temporal order is the dominant signal |
| Self-supervised successor links, adapted teacher, two-pass learned counts | 0.9190 | +0.011 | Counts learned from profiles beat fixed quotas |
| Confidence-adaptive blend of link models | 0.9237 | +0.005 | Link models are complementary |
| Eight ranked timelines with clean-margin confidence | 0.9272 | +0.004 | Rank within the query; average over hypotheses |
| Boundary consensus refiner | 0.9280 | +0.0008 | Errors concentrate at activity edges |
| Four refiner seeds | 0.9286 | +0.0007 | Seed averaging stabilises flip decisions |
| Whitened geometry with kNN temperature 0.3 (goodpjw2008's recipe) | 0.92875 | +0.0001 | Large development gain, small transfer |
| Refiner with 63 leaves | 0.92956 | +0.0008 | Not supported on development folds (−0.0003, 2/5) |
| Refiner decision threshold 0.55 | 0.92920 | +0.0005 | Not supported on development folds (−0.0003, 3/5) |
| Sixteen refiner seeds | 0.92899 | +0.0002 | Within seed noise (−0.0001, 2/5) |
| <span style="color:#999">Length-aware activity minimum</span> | 0.7322 | −0.045 | Duration rules should not depend on recording length |
| <span style="color:#999">Exercise identity in the count model</span> | 0.9221 | −0.0016 | Counts are better predicted from profile shape alone |
| <span style="color:#999">One ranked timeline instead of the blend</span> | 0.9228 | −0.0009 | A single hypothesis is too brittle |
| <span style="color:#999">Sixteen timelines instead of eight</span> | 0.9259 | −0.0013 | More perturbations add noise, not coverage |
| <span style="color:#999">Fixed 50/50 original and whitened geometry</span> | 0.9265 | −0.0023 | Mixing geometries dilutes the whitened graph |
| <span style="color:#999">Public models' probabilities as refiner features (Hanh Tran, goodpjw2008, both)</span> | 0.92782–0.92843 | −0.0003 to −0.0009 | Their evidence is already represented |
| <span style="color:#999">Count-pressure refiner features</span> | 0.92856 | −0.0002 | Count calibration is not the source of boundary errors |

Interventions rejected on the development folds against the whitened configuration (consistency requirement: a
positive pooled change and at least four positive folds):

| Intervention | Development macro-F1 | Positive folds | Derived conclusion |
|---|---:|---:|---|
| Graph smoothing 1.5 with whitening | 0.93217 | 3/5 | Gain not consistent across participants |
| Participant-balanced proposal loss | 0.93187 | 2/5 | Reweighting participants does not help |
| Analytic covariance shrinkage | 0.93177 | 1/5 | Better conditioning, not better F1 |
| Refiner prediction before context aggregation | 0.93158 | 1/5 | Aggregate first, then predict |
| Fixed semantic-supernode restart | 0.93179 | 2/5 | Rescued 321 windows, broke 324 |
| Single-flip macro-F1 regret | 0.93129 | 0/5 | Direct metric optimisation overfits |
| Whitening power 0.25 / 0.75 | 0.93036 / 0.93205 | 0/5 / 2/5 | Power 0.5 is a local optimum |
| kNN temperature 0.2 / 0.5 | 0.93129 / 0.93150 | 2/5 / 2/5 | Temperature 0.3 is a local optimum |
| Label-level blends with the public models' probabilities | 0.93043–0.93174 | ≤1/5 | Blending final labels dilutes the refiner |
| ECDF ranks of the alternative probability within participant and direction | 0.93149 | 0/5 | Relative ranks add no information beyond the raw margins |

The main open problem is localising boundaries where the evidence and the timelines disagree.

## What the public leaderboard can resolve

The public leaderboard is computed on half of the test windows of four participants. goodpjw2008 measured in
[WEAR@HASCA | Part 2: Ranked Links](https://www.kaggle.com/code/goodpjw2008/wear-hasca-part-2-ranked-links-lb-0-92990)
that reruns of identical code, and changes that move about 150 labels, shift the public score by up to ±0.005. The
+0.0008 public gain of the 63-leaf refiner lies inside that band and contradicts the development folds.

> **Selection in this notebook therefore follows the development folds, not the public leaderboard.**

## References

* Timeline reconstruction and graph: [Briano, Timeline + Graph](https://www.kaggle.com/code/woominyo/lb-0-890-wear-hasca-2026-timeline-graph) (Apache 2.0)
* Window models and fast inference: [Jiwei Liu](https://www.kaggle.com/code/jiweiliu/public-fast-gpu-inference); model weights released under Apache 2.0
* Successor links: [Hanh Tran](https://www.kaggle.com/code/honghanhhh/wear-hasca-lb-0-9)
* Learned links, counts, boundary refiner and the whitening recipe: [goodpjw2008](https://www.kaggle.com/code/goodpjw2008/wear-hasca-learned-links-counts-lb-0-92942)
* Ranked links and leaderboard-resolution measurements: [goodpjw2008, Part 2](https://www.kaggle.com/code/goodpjw2008/wear-hasca-part-2-ranked-links-lb-0-92990)

Our additions are the query-grouped ranking with clean-margin confidence, the consensus boundary features, fixed
refiner-seed averaging, and the cross-fitting audit above.